Loading the Odds Data for only Home Team and Away Team and the respective scorlines for the fixured from 2016

In [1]:
import pandas as pd
import numpy as np

BASE = r"C:\Users\veers\OneDrive\Documents\FPL Agent\fpl-copilot"
odds = pd.read_parquet(BASE + r"\data\history\odds_all_seasons.parquet")

print("Shape:", odds.shape)
cols = [c for c in ["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR"] if c in odds.columns]
print(odds[cols].head(10).to_string(index=False))

Shape: (3800, 181)
    Date       HomeTeam   AwayTeam  FTHG  FTAG FTR
13/08/16        Burnley    Swansea     0     1   A
13/08/16 Crystal Palace  West Brom     0     1   A
13/08/16        Everton  Tottenham     1     1   D
13/08/16           Hull  Leicester     2     1   H
13/08/16       Man City Sunderland     2     1   H
13/08/16  Middlesbrough      Stoke     1     1   D
13/08/16    Southampton    Watford     1     1   D
14/08/16        Arsenal  Liverpool     3     4   A
14/08/16    Bournemouth Man United     1     3   A
15/08/16        Chelsea   West Ham     2     1   H


In [2]:
# === Shape match data for Dixon-Coles ===
matches = odds[["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG"]].copy()
matches.columns = ["date", "home", "away", "home_goals", "away_goals"]

# Basic sanity checks
print("Total matches:", len(matches))
print("Unique teams:", matches["home"].nunique())
print("Any missing values?\n", matches.isna().sum())
print("\nGoal ranges — home:", matches["home_goals"].min(), "to", matches["home_goals"].max())
print("Goal ranges — away:", matches["away_goals"].min(), "to", matches["away_goals"].max())
print("\nAvg home goals:", round(matches["home_goals"].mean(), 3))
print("Avg away goals:", round(matches["away_goals"].mean(), 3))

Total matches: 3800
Unique teams: 34
Any missing values?
 date          0
home          0
away          0
home_goals    0
away_goals    0
dtype: int64

Goal ranges — home: 0 to 9
Goal ranges — away: 0 to 9

Avg home goals: 1.555
Avg away goals: 1.28


Giving all teams an ID and finding how many unique teams are there over the years we are training on

In [3]:
# === Map teams to array positions (the optimizer needs numbers, not names) ===
teams = sorted(set(matches["home"]) | set(matches["away"]))
team_idx = {team: i for i, team in enumerate(teams)}
n_teams = len(teams)

print("Number of teams:", n_teams)
print("Total parameters we'll fit:", n_teams * 2 + 1, "  (", n_teams, "attacks +", n_teams, "defences + 1 home)")

# Add index columns to the matches frame
matches["home_i"] = matches["home"].map(team_idx)
matches["away_i"] = matches["away"].map(team_idx)

print("\nFirst few rows with indices:")
print(matches[["home", "home_i", "away", "away_i", "home_goals", "away_goals"]].head().to_string(index=False))

Number of teams: 34
Total parameters we'll fit: 69   ( 34 attacks + 34 defences + 1 home)

First few rows with indices:
          home  home_i       away  away_i  home_goals  away_goals
       Burnley       5    Swansea      28           0           1
Crystal Palace       8  West Brom      31           0           1
       Everton       9  Tottenham      29           1           1
          Hull      12  Leicester      15           2           1
      Man City      18 Sunderland      27           2           1


Funtion for the Poisson Dist, to get the attack, defence and home ground strengths respectively

In [4]:
from scipy.stats import poisson

def neg_log_likelihood(params, matches, n_teams):
    # 1-2. Unpack the flat array into meaningful pieces
    attack = params[:n_teams]          # first 34 = attack strengths
    defence = params[n_teams:2*n_teams] # next 34 = defence weaknesses
    home_adv = params[-1]               # last one = home advantage

    # 3. Expected goals for every match at once (vectorised)
    h, a = matches["home_i"].values, matches["away_i"].values
    lambda_home = np.exp(attack[h] + defence[a] + home_adv)
    lambda_away = np.exp(attack[a] + defence[h])

    # 4. Poisson log-probability of the actual scores
    ll_home = poisson.logpmf(matches["home_goals"].values, lambda_home)
    ll_away = poisson.logpmf(matches["away_goals"].values, lambda_away)

    # 5. Sum everything, return the NEGATIVE (optimizer minimizes)
    return -(ll_home.sum() + ll_away.sum())

print("Function defined.")

Function defined.


Optimiser function to maximize the Likelihood

In [5]:
from scipy.optimize import minimize

# Starting guess: all strengths at 0, home advantage at a small positive value.
# (Because we use exp(), 0 means "average" — exp(0)=1, a neutral multiplier.)
x0 = np.zeros(2 * n_teams + 1)
x0[-1] = 0.25   # a sensible starting home advantage

print("Fitting", len(x0), "parameters on", len(matches), "matches...")

result = minimize(
    neg_log_likelihood,
    x0,
    args=(matches, n_teams),
    method="L-BFGS-B",   # a standard, fast optimizer for this kind of problem
)

print("\nConverged:", result.success)
print("Final negative log-likelihood:", round(result.fun, 1))
print("Message:", result.message)

Fitting 69 parameters on 3800 matches...

Converged: True
Final negative log-likelihood: 11128.3
Message: CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH


Team Strengths over the years

In [6]:
# === Extract and inspect the fitted team strengths ===
attack = result.x[:n_teams]
defence = result.x[n_teams:2*n_teams]
home_adv = result.x[-1]

strengths = pd.DataFrame({
    "team": teams,
    "attack": attack.round(3),
    "defence": defence.round(3),
}).sort_values("attack", ascending=False)

print("Home advantage:", round(home_adv, 3))
print("\n=== Top 8 attacks (higher = scores more) ===")
print(strengths.head(8).to_string(index=False))
print("\n=== Best 8 defences (LOWER = concedes less) ===")
print(strengths.sort_values("defence").head(8).to_string(index=False))

Home advantage: 0.195

=== Top 8 attacks (higher = scores more) ===
       team  attack  defence
   Man City   0.743   -0.458
  Liverpool   0.636   -0.307
    Arsenal   0.517   -0.243
  Tottenham   0.474   -0.100
    Chelsea   0.423   -0.193
 Man United   0.363   -0.163
  Brentford   0.310    0.002
Aston Villa   0.286   -0.021

=== Best 8 defences (LOWER = concedes less) ===
      team  attack  defence
  Man City   0.743   -0.458
 Liverpool   0.636   -0.307
   Arsenal   0.517   -0.243
   Chelsea   0.423   -0.193
Man United   0.363   -0.163
 Tottenham   0.474   -0.100
   Everton   0.078   -0.040
 Newcastle   0.245   -0.027


Predicting a certain game

In [7]:
# === Predict a single fixture from the fitted strengths ===
def predict_fixture(home_team, away_team, max_goals=6):
    hi, ai = team_idx[home_team], team_idx[away_team]
    lam_home = np.exp(attack[hi] + defence[ai] + home_adv)
    lam_away = np.exp(attack[ai] + defence[hi])

    # Probability of each goal count (0..max_goals) for each side
    h_probs = poisson.pmf(np.arange(max_goals + 1), lam_home)
    a_probs = poisson.pmf(np.arange(max_goals + 1), lam_away)

    # Scoreline matrix: P(home=i AND away=j) = outer product (independence)
    score_matrix = np.outer(h_probs, a_probs)

    # Derived outcomes
    p_home_win = np.tril(score_matrix, -1).sum()   # home > away
    p_draw     = np.trace(score_matrix)            # home == away
    p_away_win = np.triu(score_matrix, 1).sum()    # away > home
    p_home_cs  = a_probs[0]                         # away scores 0

    print(f"{home_team} (home) vs {away_team}")
    print(f"  Expected goals:  {home_team} {lam_home:.2f}  -  {lam_away:.2f} {away_team}")
    print(f"  Home win: {p_home_win:.1%} | Draw: {p_draw:.1%} | Away win: {p_away_win:.1%}")
    print(f"  Home clean sheet: {p_home_cs:.1%}")
    return lam_home, lam_away

predict_fixture("Man City", "Everton")
print()
predict_fixture("Everton", "Man City")


Man City (home) vs Everton
  Expected goals:  Man City 2.46  -  0.68 Everton
  Home win: 75.0% | Draw: 15.3% | Away win: 8.4%
  Home clean sheet: 50.5%

Everton (home) vs Man City
  Expected goals:  Everton 0.83  -  2.02 Man City
  Home win: 14.5% | Draw: 20.4% | Away win: 64.6%
  Home clean sheet: 13.3%


(np.float64(0.8313765530623203), np.float64(2.0207127591409217))

In [8]:
# === Add season column — handle mixed date formats ===
matches["date_parsed"] = pd.to_datetime(matches["date"], dayfirst=True)

def to_season(d):
    start_year = d.year if d.month >= 8 else d.year - 1
    return f"{start_year}-{str(start_year + 1)[2:]}"

matches["season"] = matches["date_parsed"].apply(to_season)

print("Any dates that failed to parse:", matches["date_parsed"].isna().sum())
print("\nMatches per season:")
print(matches["season"].value_counts().sort_index())

Any dates that failed to parse: 0

Matches per season:
season
2016-17    380
2017-18    380
2018-19    380
2019-20    380
2020-21    380
2021-22    380
2022-23    380
2023-24    380
2024-25    380
2025-26    380
Name: count, dtype: int64


C:\Users\veers\AppData\Local\Temp\ipykernel_20644\115868130.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  matches["date_parsed"] = pd.to_datetime(matches["date"], dayfirst=True)


Fitting the Dixon Coles Model

In [9]:
# === Fit Dixon-Coles on a subset of matches ===
def fit_dixon_coles(train_matches, all_teams):
    idx = {t: i for i, t in enumerate(all_teams)}
    nt = len(all_teams)
    h = train_matches["home"].map(idx).values
    a = train_matches["away"].map(idx).values
    hg = train_matches["home_goals"].values
    ag = train_matches["away_goals"].values

    def nll(params):
        atk, dfc, hadv = params[:nt], params[nt:2*nt], params[-1]
        lam_h = np.exp(atk[h] + dfc[a] + hadv)
        lam_a = np.exp(atk[a] + dfc[h])
        return -(poisson.logpmf(hg, lam_h).sum() + poisson.logpmf(ag, lam_a).sum())

    x0 = np.zeros(2 * nt + 1)
    x0[-1] = 0.25
    res = minimize(nll, x0, method="L-BFGS-B")
    return res.x, idx, nt

# Fit on train seasons only
train_seasons = ["2016-17","2017-18","2018-19","2019-20","2020-21","2021-22","2022-23","2023-24"]
train_m = matches[matches["season"].isin(train_seasons)]
val_m   = matches[matches["season"] == "2024-25"]

params_fit, idx_fit, nt_fit = fit_dixon_coles(train_m, teams)
print("Fitted on", len(train_m), "matches. Converged.")

Fitted on 3040 matches. Converged.


Avg of Log Likelihoods

In [10]:
# === Evaluate on validation season vs baselines ===
def avg_log_likelihood(eval_m, params, idx, nt):
    # Skip matches with teams not seen in training (promoted sides)
    known = eval_m["home"].isin(idx) & eval_m["away"].isin(idx)
    m = eval_m[known]
    atk, dfc, hadv = params[:nt], params[nt:2*nt], params[-1]
    h = m["home"].map(idx).values
    a = m["away"].map(idx).values
    lam_h = np.exp(atk[h] + dfc[a] + hadv)
    lam_a = np.exp(atk[a] + dfc[h])
    ll = poisson.logpmf(m["home_goals"].values, lam_h) + poisson.logpmf(m["away_goals"].values, lam_a)
    return ll.mean(), len(m), (~known).sum()

# Our model
dc_ll, n_scored, n_skipped = avg_log_likelihood(val_m, params_fit, idx_fit, nt_fit)

# Baseline 1: home-advantage only (all teams average -> atk=dfc=0)
flat = np.zeros(2 * nt_fit + 1)
flat[-1] = params_fit[-1]   # keep the fitted home advantage
base_ll, _, _ = avg_log_likelihood(val_m, flat, idx_fit, nt_fit)

# Baseline 2: league-average goals (fit only the global scoring rate)
avg_h = np.log(train_m["home_goals"].mean())
avg_a = np.log(train_m["away_goals"].mean())
flat2 = np.zeros(2 * nt_fit + 1)
# encode avg rates via home_adv trick: set home_adv so exp = avg_h, defence baseline = avg_a
# simplest: compute directly
m2 = val_m[val_m["home"].isin(idx_fit) & val_m["away"].isin(idx_fit)]
ll2 = (poisson.logpmf(m2["home_goals"].values, train_m["home_goals"].mean())
       + poisson.logpmf(m2["away_goals"].values, train_m["away_goals"].mean())).mean()

print(f"Matches scored: {n_scored}  (skipped {n_skipped} with unseen promoted teams)\n")
print("Avg log-likelihood per match (higher = better):")
print(f"  Dixon-Coles          : {dc_ll:.4f}")
print(f"  Home-advantage only  : {base_ll:.4f}")
print(f"  League-average goals : {ll2:.4f}")

Matches scored: 380  (skipped 0 with unseen promoted teams)

Avg log-likelihood per match (higher = better):
  Dixon-Coles          : -3.0595
  Home-advantage only  : -3.1780
  League-average goals : -3.0795


In [11]:
# === Check: were any 2024-25 teams genuinely absent from training? ===
val_teams = set(val_m["home"]) | set(val_m["away"])
train_teams = set(train_m["home"]) | set(train_m["away"])

print("Teams in 2024-25:", len(val_teams))
print("Teams in training:", len(train_teams))

promoted = val_teams - train_teams
print("\n2024-25 teams NOT in training (expected ~3 promoted):")
print(promoted if promoted else "NONE — suspicious")

# Sanity: which teams got promoted to 2024-25? Ipswich, Leicester, Southampton
# Leicester & Southampton have PL history; Ipswich likely genuinely new
print("\nAll 2024-25 teams:")
print(sorted(val_teams))

Teams in 2024-25: 20
Teams in training: 33

2024-25 teams NOT in training (expected ~3 promoted):
{'Ipswich'}

All 2024-25 teams:
['Arsenal', 'Aston Villa', 'Bournemouth', 'Brentford', 'Brighton', 'Chelsea', 'Crystal Palace', 'Everton', 'Fulham', 'Ipswich', 'Leicester', 'Liverpool', 'Man City', 'Man United', 'Newcastle', "Nott'm Forest", 'Southampton', 'Tottenham', 'West Ham', 'Wolves']


Dixon Cole's Low Score Correction

In [12]:
# === Dixon-Coles with the low-score correction (tau) ===
def fit_dixon_coles_dc(train_matches, all_teams):
    idx = {t: i for i, t in enumerate(all_teams)}
    nt = len(all_teams)
    h  = train_matches["home"].map(idx).values
    a  = train_matches["away"].map(idx).values
    hg = train_matches["home_goals"].values
    ag = train_matches["away_goals"].values

    def nll(params):
        atk, dfc = params[:nt], params[nt:2*nt]
        hadv, rho = params[-2], params[-1]
        lam_h = np.exp(atk[h] + dfc[a] + hadv)
        lam_a = np.exp(atk[a] + dfc[h])

        # Base Poisson log-probs
        log_p = poisson.logpmf(hg, lam_h) + poisson.logpmf(ag, lam_a)

        # Tau correction, applied only to the 4 low scorelines
        tau = np.ones(len(hg))
        tau[(hg==0)&(ag==0)] = (1 - lam_h*lam_a*rho)[(hg==0)&(ag==0)]
        tau[(hg==0)&(ag==1)] = (1 + lam_h*rho)[(hg==0)&(ag==1)]
        tau[(hg==1)&(ag==0)] = (1 + lam_a*rho)[(hg==1)&(ag==0)]
        tau[(hg==1)&(ag==1)] = (1 - rho)

        # tau must stay positive (log needs it); clip tiny/negative values
        tau = np.clip(tau, 1e-10, None)
        return -(log_p + np.log(tau)).sum()

    x0 = np.zeros(2*nt + 2)
    x0[-2] = 0.25   # home advantage
    x0[-1] = 0.0    # rho starts at 0 (= plain Poisson)
    res = minimize(nll, x0, method="L-BFGS-B")
    return res.x, idx, nt

params_dc, idx_dc, nt_dc = fit_dixon_coles_dc(train_m, teams)
print("Converged. Rho (low-score correction):", round(params_dc[-1], 4))
print("Home advantage:", round(params_dc[-2], 4))

Converged. Rho (low-score correction): -0.0292
Home advantage: 0.2095


Log Likelihoods respectively

In [13]:
# === Does the DC correction beat plain Poisson on validation? ===
def score_dc(eval_m, params, idx, nt, use_tau=True):
    m = eval_m[eval_m["home"].isin(idx) & eval_m["away"].isin(idx)]
    atk, dfc = params[:nt], params[nt:2*nt]
    hadv, rho = params[-2], params[-1]
    h = m["home"].map(idx).values; a = m["away"].map(idx).values
    hg = m["home_goals"].values;   ag = m["away_goals"].values
    lam_h = np.exp(atk[h] + dfc[a] + hadv)
    lam_a = np.exp(atk[a] + dfc[h])
    log_p = poisson.logpmf(hg, lam_h) + poisson.logpmf(ag, lam_a)
    if use_tau:
        tau = np.ones(len(hg))
        tau[(hg==0)&(ag==0)] = (1 - lam_h*lam_a*rho)[(hg==0)&(ag==0)]
        tau[(hg==0)&(ag==1)] = (1 + lam_h*rho)[(hg==0)&(ag==1)]
        tau[(hg==1)&(ag==0)] = (1 + lam_a*rho)[(hg==1)&(ag==0)]
        tau[(hg==1)&(ag==1)] = (1 - rho)
        log_p = log_p + np.log(np.clip(tau, 1e-10, None))
    return log_p.mean()

with_tau = score_dc(val_m, params_dc, idx_dc, nt_dc, use_tau=True)
without  = score_dc(val_m, params_dc, idx_dc, nt_dc, use_tau=False)

print("Avg log-likelihood on validation (higher = better):")
print(f"  With DC correction   : {with_tau:.4f}")
print(f"  Without (plain Poisson): {without:.4f}")
print(f"  Plain Poisson (§earlier): -3.0595")

Avg log-likelihood on validation (higher = better):
  With DC correction   : -3.0596
  Without (plain Poisson): -3.0594
  Plain Poisson (§earlier): -3.0595


Time Decay with Dixon Coles

In [14]:
# === Dixon-Coles with time-decay ===
def fit_dc_decay(train_matches, all_teams, ref_date, half_life_days):
    idx = {t: i for i, t in enumerate(all_teams)}
    nt = len(all_teams)
    h  = train_matches["home"].map(idx).values
    a  = train_matches["away"].map(idx).values
    hg = train_matches["home_goals"].values
    ag = train_matches["away_goals"].values

    # Age in days before the reference date, then decay weight
    age = (ref_date - train_matches["date_parsed"]).dt.days.values
    if half_life_days is None:          # no decay
        w = np.ones(len(age))
    else:
        xi = np.log(2) / half_life_days  # convert half-life -> decay rate
        w = np.exp(-xi * age)

    def nll(params):
        atk, dfc = params[:nt], params[nt:2*nt]
        hadv, rho = params[-2], params[-1]
        lam_h = np.exp(atk[h] + dfc[a] + hadv)
        lam_a = np.exp(atk[a] + dfc[h])
        log_p = poisson.logpmf(hg, lam_h) + poisson.logpmf(ag, lam_a)
        tau = np.ones(len(hg))
        tau[(hg==0)&(ag==0)] = (1 - lam_h*lam_a*rho)[(hg==0)&(ag==0)]
        tau[(hg==0)&(ag==1)] = (1 + lam_h*rho)[(hg==0)&(ag==1)]
        tau[(hg==1)&(ag==0)] = (1 + lam_a*rho)[(hg==1)&(ag==0)]
        tau[(hg==1)&(ag==1)] = (1 - rho)
        log_p = log_p + np.log(np.clip(tau, 1e-10, None))
        return -(w * log_p).sum()        # weighted by recency

    x0 = np.zeros(2*nt + 2); x0[-2] = 0.25
    res = minimize(nll, x0, method="L-BFGS-B")
    return res.x, idx, nt

# Reference = start of the validation season (predict 2024-25 using only prior info)
ref = matches[matches["season"] == "2024-25"]["date_parsed"].min()
print("Reference date (val season start):", ref.date())
print("Function ready.")

Reference date (val season start): 2024-08-16
Function ready.


GridSearch to find the best time for the Time Decay

In [15]:
# === Gridsearch the half-life ===
half_lives = {
    "no decay (inf)": None,
    "2 years (730d)": 730,
    "18 months (540d)": 540,
    "1 year (365d)": 365,
    "6 months (180d)": 180,
}

print("Half-life           Val log-likelihood (higher = better)")
print("-" * 52)
for label, hl in half_lives.items():
    params, idx, nt = fit_dc_decay(train_m, teams, ref, hl)
    ll = score_dc(val_m, params, idx, nt, use_tau=True)
    print(f"{label:20s} {ll:.4f}")

Half-life           Val log-likelihood (higher = better)
----------------------------------------------------
no decay (inf)       -3.0596
2 years (730d)       -3.0234
18 months (540d)     -3.0156
1 year (365d)        -3.0068
6 months (180d)      -3.0088


Evaluating on Win, Loss and Draw

In [16]:
# === Evaluate on Win/Draw/Loss (more interpretable than exact score) ===
def score_wdl(eval_m, params, idx, nt, max_goals=10):
    m = eval_m[eval_m["home"].isin(idx) & eval_m["away"].isin(idx)]
    atk, dfc, hadv = params[:nt], params[nt:2*nt], params[-2]
    h = m["home"].map(idx).values; a = m["away"].map(idx).values
    hg = m["home_goals"].values;   ag = m["away_goals"].values

    correct, probs_on_actual = 0, []
    for i in range(len(m)):
        lam_h = np.exp(atk[h[i]] + dfc[a[i]] + hadv)
        lam_a = np.exp(atk[a[i]] + dfc[h[i]])
        hp = poisson.pmf(np.arange(max_goals+1), lam_h)
        ap = poisson.pmf(np.arange(max_goals+1), lam_a)
        M = np.outer(hp, ap)
        p_home = np.tril(M, -1).sum()
        p_draw = np.trace(M)
        p_away = np.triu(M, 1).sum()

        # actual outcome
        actual = "H" if hg[i] > ag[i] else ("A" if ag[i] > hg[i] else "D")
        pred   = ["H","D","A"][np.argmax([p_home, p_draw, p_away])]
        if pred == actual:
            correct += 1
        probs_on_actual.append({"H":p_home,"D":p_draw,"A":p_away}[actual])

    return correct/len(m), np.mean(probs_on_actual), len(m)

# Refit at the winning half-life (1 year) and score
params_1y, idx_1y, nt_1y = fit_dc_decay(train_m, teams, ref, 365)
acc, avg_p, n = score_wdl(val_m, params_1y, idx_1y, nt_1y)

print(f"Validation matches: {n}")
print(f"Top-pick accuracy (predicted outcome = actual): {acc:.1%}")
print(f"Avg probability model put on the ACTUAL outcome: {avg_p:.1%}")
print(f"\nFor reference — always guessing 'home win': {(val_m['home_goals'] > val_m['away_goals']).mean():.1%}")

Validation matches: 380
Top-pick accuracy (predicted outcome = actual): 48.9%
Avg probability model put on the ACTUAL outcome: 40.2%

For reference — always guessing 'home win': 40.8%


Blending the actual Odds data features into our model

In [17]:
# === Extract clean (vig-free) outcome probabilities from bookmaker odds ===
# B365H/D/A = Bet365 decimal odds for home / draw / away
odds_cols = ["B365H", "B365D", "B365A"]
print("Odds columns present:", [c for c in odds_cols if c in odds.columns])
print("Missing values in odds:", odds[odds_cols].isna().sum().to_dict())

# Attach odds to our matches frame (same row order, same source file)
m = matches.copy()
for c in odds_cols:
    m[c] = odds[c].values

# Step 1: invert odds -> raw implied probabilities
m["raw_H"] = 1 / m["B365H"]
m["raw_D"] = 1 / m["B365D"]
m["raw_A"] = 1 / m["B365A"]

# Step 2: the overround (how much >100% they sum to)
m["overround"] = m["raw_H"] + m["raw_D"] + m["raw_A"]

# Step 3: normalise to strip the vig
m["p_H"] = m["raw_H"] / m["overround"]
m["p_D"] = m["raw_D"] / m["overround"]
m["p_A"] = m["raw_A"] / m["overround"]

print("\nAvg bookmaker overround:", round(m["overround"].mean(), 4), "(1.05 = 5% margin)")
print("\nSample — first 5 matches:")
print(m[["home","away","B365H","B365D","B365A","p_H","p_D","p_A"]].head().to_string(index=False))
print("\nCheck probabilities sum to 1:", round(m[["p_H","p_D","p_A"]].sum(axis=1).mean(), 6))

Odds columns present: ['B365H', 'B365D', 'B365A']
Missing values in odds: {'B365H': 0, 'B365D': 0, 'B365A': 0}

Avg bookmaker overround: 1.0468 (1.05 = 5% margin)

Sample — first 5 matches:
          home       away  B365H  B365D  B365A      p_H      p_D      p_A
       Burnley    Swansea   2.40    3.3   3.25 0.405559 0.294952 0.299490
Crystal Palace  West Brom   2.00    3.3   4.50 0.487685 0.295567 0.216749
       Everton  Tottenham   3.20    3.4   2.40 0.305389 0.287425 0.407186
          Hull  Leicester   4.50    3.6   1.91 0.217107 0.271384 0.511509
      Man City Sunderland   1.25    6.5  15.00 0.783920 0.150754 0.065327

Check probabilities sum to 1: 1.0


Removing Lambda from the probabilites

In [18]:
# === Invert market outcome probabilities -> implied (lambda_home, lambda_away) ===
from scipy.optimize import minimize as sp_min

def outcomes_from_lambdas(lam_h, lam_a, max_goals=10):
    hp = poisson.pmf(np.arange(max_goals+1), lam_h)
    ap = poisson.pmf(np.arange(max_goals+1), lam_a)
    M = np.outer(hp, ap)
    return np.tril(M, -1).sum(), np.trace(M), np.triu(M, 1).sum()  # H, D, A

def implied_lambdas(pH, pD, pA):
    # search for the (lam_h, lam_a) whose forward outcomes match the market
    def mismatch(log_lams):
        lam_h, lam_a = np.exp(log_lams)          # exp keeps them positive
        mH, mD, mA = outcomes_from_lambdas(lam_h, lam_a)
        return (mH-pH)**2 + (mD-pD)**2 + (mA-pA)**2
    res = sp_min(mismatch, x0=[np.log(1.4), np.log(1.1)], method="Nelder-Mead")
    return np.exp(res.x)

# Run it for every match (this takes a minute — 3,800 small optimizations)
print("Inverting odds for", len(m), "matches...")
lam_pairs = np.array([
    implied_lambdas(r.p_H, r.p_D, r.p_A)
    for r in m.itertuples()
])
m["mkt_lam_h"] = lam_pairs[:, 0]
m["mkt_lam_a"] = lam_pairs[:, 1]

print("Done.")
print("\nMarket-implied goal expectations — sample:")
print(m[["home","away","p_H","p_D","p_A","mkt_lam_h","mkt_lam_a"]].head().to_string(index=False))
print("\nAvg market lambda — home:", round(m['mkt_lam_h'].mean(),3), " away:", round(m['mkt_lam_a'].mean(),3))

Inverting odds for 3800 matches...
Done.

Market-implied goal expectations — sample:
          home       away      p_H      p_D      p_A  mkt_lam_h  mkt_lam_a
       Burnley    Swansea 0.405559 0.294952 0.299490   1.168507   0.960257
Crystal Palace  West Brom 0.487685 0.295567 0.216749   1.252298   0.729935
       Everton  Tottenham 0.305389 0.287425 0.407186   1.010509   1.214076
          Hull  Leicester 0.217107 0.271384 0.511509   0.821545   1.420973
      Man City Sunderland 0.783920 0.150754 0.065327   2.346310   0.521485

Avg market lambda — home: 1.449  away: 1.148


Combining Dixon Coles and Odds

In [19]:
# === Gridsearch the blend weight w ===
# DC lambdas on validation, using the 1-year-decay fit
val = m[m["season"] == "2024-25"].copy()
known = val["home"].isin(idx_1y) & val["away"].isin(idx_1y)
val = val[known].copy()

atk, dfc, hadv = params_1y[:nt_1y], params_1y[nt_1y:2*nt_1y], params_1y[-2]
hi = val["home"].map(idx_1y).values
ai = val["away"].map(idx_1y).values
val["dc_lam_h"] = np.exp(atk[hi] + dfc[ai] + hadv)
val["dc_lam_a"] = np.exp(atk[ai] + dfc[hi])

def wdl_from_lambdas(lam_h, lam_a, actual, max_goals=10):
    mH, mD, mA = outcomes_from_lambdas(lam_h, lam_a, max_goals)
    pred = ["H","D","A"][np.argmax([mH, mD, mA])]
    p_actual = {"H":mH,"D":mD,"A":mA}[actual]
    return pred == actual, p_actual

val["actual"] = np.where(val["home_goals"] > val["away_goals"], "H",
                 np.where(val["away_goals"] > val["home_goals"], "A", "D"))

print("  w     DC-weight   accuracy   avg P(actual)")
print("-" * 46)
for w in [0.0, 0.2, 0.4, 0.5, 0.6, 0.8, 1.0]:
    lam_h = w * val["dc_lam_h"] + (1-w) * val["mkt_lam_h"]
    lam_a = w * val["dc_lam_a"] + (1-w) * val["mkt_lam_a"]
    results = [wdl_from_lambdas(lh, la, ac) for lh, la, ac in zip(lam_h, lam_a, val["actual"])]
    acc = np.mean([r[0] for r in results])
    pact = np.mean([r[1] for r in results])
    tag = "pure market" if w==0 else ("pure DC" if w==1 else "")
    print(f" {w:.1f}      {w:.1f}       {acc:.1%}      {pact:.1%}   {tag}")

  w     DC-weight   accuracy   avg P(actual)
----------------------------------------------
 0.0      0.0       53.7%      42.3%   pure market
 0.2      0.2       52.6%      41.9%   
 0.4      0.4       52.6%      41.5%   
 0.5      0.5       51.8%      41.3%   
 0.6      0.6       50.8%      41.1%   
 0.8      0.8       50.0%      40.6%   
 1.0      1.0       48.9%      40.2%   pure DC


Does Dixon Coles contribute to predicting clean sheets?

In [20]:
# === Which model predicts CLEAN SHEETS best? ===
# Actual clean sheets in validation
val["home_cs_actual"] = (val["away_goals"] == 0).astype(int)
val["away_cs_actual"] = (val["home_goals"] == 0).astype(int)

def cs_logloss(lam_h, lam_a):
    # P(home CS) = P(away scores 0) = exp(-lam_a); and vice versa
    p_home_cs = np.exp(-lam_a)
    p_away_cs = np.exp(-lam_h)
    # Brier score against actual (lower = better)
    b_home = ((p_home_cs - val["home_cs_actual"])**2).mean()
    b_away = ((p_away_cs - val["away_cs_actual"])**2).mean()
    return (b_home + b_away) / 2

print("Clean-sheet Brier score (lower = better):")
print("-" * 40)
for w in [0.0, 0.2, 0.4, 0.5, 0.6, 0.8, 1.0]:
    lam_h = w * val["dc_lam_h"] + (1-w) * val["mkt_lam_h"]
    lam_a = w * val["dc_lam_a"] + (1-w) * val["mkt_lam_a"]
    tag = "pure market" if w==0 else ("pure DC" if w==1 else "")
    print(f"  w={w:.1f}  Brier {cs_logloss(lam_h, lam_a):.4f}   {tag}")

# Baseline: always predict the base clean-sheet rate
base_cs = val[["home_cs_actual","away_cs_actual"]].values.mean()
print(f"\nBase clean-sheet rate: {base_cs:.1%}")
print(f"Baseline Brier (always predict {base_cs:.2f}): "
      f"{((base_cs - val[['home_cs_actual','away_cs_actual']].values)**2).mean():.4f}")

Clean-sheet Brier score (lower = better):
----------------------------------------
  w=0.0  Brier 0.1723   pure market
  w=0.2  Brier 0.1718   
  w=0.4  Brier 0.1719   
  w=0.5  Brier 0.1722   
  w=0.6  Brier 0.1726   
  w=0.8  Brier 0.1737   
  w=1.0  Brier 0.1753   pure DC

Base clean-sheet rate: 23.4%
Baseline Brier (always predict 0.23): 0.1794


Does Home Advantage actually matter?

In [21]:
# === Experiment 1: per-team home advantage vs global ===
def fit_dc_per_team_home(train_matches, all_teams, ref_date, half_life_days):
    idx = {t: i for i, t in enumerate(all_teams)}
    nt = len(all_teams)
    h  = train_matches["home"].map(idx).values
    a  = train_matches["away"].map(idx).values
    hg = train_matches["home_goals"].values
    ag = train_matches["away_goals"].values
    age = (ref_date - train_matches["date_parsed"]).dt.days.values
    xi = np.log(2)/half_life_days
    w = np.exp(-xi * age)

    # params: [attack(nt), defence(nt), home_adv_per_team(nt), rho]
    def nll(params):
        atk = params[:nt]
        dfc = params[nt:2*nt]
        hadv = params[2*nt:3*nt]        # per-team now
        rho = params[-1]
        lam_h = np.exp(atk[h] + dfc[a] + hadv[h])   # home team's own advantage
        lam_a = np.exp(atk[a] + dfc[h])
        log_p = poisson.logpmf(hg, lam_h) + poisson.logpmf(ag, lam_a)
        tau = np.ones(len(hg))
        tau[(hg==0)&(ag==0)] = (1 - lam_h*lam_a*rho)[(hg==0)&(ag==0)]
        tau[(hg==0)&(ag==1)] = (1 + lam_h*rho)[(hg==0)&(ag==1)]
        tau[(hg==1)&(ag==0)] = (1 + lam_a*rho)[(hg==1)&(ag==0)]
        tau[(hg==1)&(ag==1)] = (1 - rho)
        log_p = log_p + np.log(np.clip(tau, 1e-10, None))
        return -(w * log_p).sum()

    x0 = np.zeros(3*nt + 1); x0[2*nt:3*nt] = 0.25
    res = minimize(nll, x0, method="L-BFGS-B")
    return res.x, idx, nt

# Fit and score on WDL
p_pt, idx_pt, nt_pt = fit_dc_per_team_home(train_m, teams, ref, 365)

# score_wdl expects params[-2] as global home_adv; per-team differs, so score inline
def score_wdl_pt(eval_m, params, idx, nt):
    mm = eval_m[eval_m["home"].isin(idx) & eval_m["away"].isin(idx)]
    atk, dfc, hadv = params[:nt], params[nt:2*nt], params[2*nt:3*nt]
    correct = 0
    for r in mm.itertuples():
        hi, ai = idx[r.home], idx[r.away]
        lh = np.exp(atk[hi] + dfc[ai] + hadv[hi])
        la = np.exp(atk[ai] + dfc[hi])
        mH, mD, mA = outcomes_from_lambdas(lh, la)
        pred = ["H","D","A"][np.argmax([mH,mD,mA])]
        actual = "H" if r.home_goals>r.away_goals else ("A" if r.away_goals>r.home_goals else "D")
        correct += (pred==actual)
    return correct/len(mm)

acc_pt = score_wdl_pt(val_m, p_pt, idx_pt, nt_pt)
print(f"Per-team home advantage — WDL accuracy: {acc_pt:.1%}")
print(f"Global home advantage   — WDL accuracy: 48.9%")
print(f"\nHome-advantage spread: {p_pt[2*nt_pt:3*nt_pt].min():.2f} to {p_pt[2*nt_pt:3*nt_pt].max():.2f}")

Per-team home advantage — WDL accuracy: 48.4%
Global home advantage   — WDL accuracy: 48.9%

Home-advantage spread: -0.13 to 1.07


Do past fixtures actually matter?

In [22]:
# === Experiment 2: are there systematic fixture-pairing effects? ===
# Use the full-data DC fit (params_dc from earlier) to get predicted goals per match
atk_a, dfc_a, hadv_a = params_dc[:nt_dc], params_dc[nt_dc:2*nt_dc], params_dc[-2]

allm = matches.copy()
allm = allm[allm["home"].isin(idx_dc) & allm["away"].isin(idx_dc)]
hi = allm["home"].map(idx_dc).values
ai = allm["away"].map(idx_dc).values
allm["pred_h"] = np.exp(atk_a[hi] + dfc_a[ai] + hadv_a)
allm["pred_a"] = np.exp(atk_a[ai] + dfc_a[hi])

# Residual = actual - predicted goals (home side)
allm["resid_h"] = allm["home_goals"] - allm["pred_h"]

# For each ordered pairing, average residual and how many meetings
pair = (allm.groupby(["home","away"])
        .agg(avg_resid=("resid_h","mean"), n=("resid_h","size"))
        .reset_index())

# Only pairings with enough meetings to say anything (>=6 over 10 seasons)
pair = pair[pair["n"] >= 6].copy()
print("Pairings with >=6 meetings:", len(pair))
print("\nMost NEGATIVE residuals (home scores fewer than predicted vs this opponent):")
print(pair.sort_values("avg_resid").head(6).to_string(index=False))
print("\nMost POSITIVE residuals (home scores more than predicted):")
print(pair.sort_values("avg_resid", ascending=False).head(6).to_string(index=False))

# The real test: is the SPREAD of pairing residuals bigger than random noise?
print(f"\nStd of pairing avg-residuals: {pair['avg_resid'].std():.3f}")
print(f"Expected std if pure noise:   {(allm['resid_h'].std() / np.sqrt(pair['n'].mean())):.3f}")

Pairings with >=6 meetings: 288

Most NEGATIVE residuals (home scores fewer than predicted vs this opponent):
          home        away  avg_resid  n
       Everton Aston Villa  -1.051360  7
Crystal Palace      Fulham  -0.991777  6
    Man United      Wolves  -0.898353  8
     Liverpool      Fulham  -0.816748  6
       Chelsea      Fulham  -0.801645  6
   Bournemouth Southampton  -0.783322  6

Most POSITIVE residuals (home scores more than predicted):
          home        away  avg_resid  n
   Aston Villa   Liverpool   1.589368  7
Crystal Palace Aston Villa   1.100018  7
        Fulham    Brighton   1.098034  6
     Newcastle   Tottenham   0.947877  9
      Man City     Burnley   0.928986  8
      Brighton      Wolves   0.906917  8

Std of pairing avg-residuals: 0.407
Expected std if pure noise:   0.423


In [23]:
# === Deeper fixture-pairing tests (A: pooled, C: significance, D: persistence) ===

# --- A: pool home+away into unordered rivalries ---
allm["pair_key"] = allm.apply(lambda r: tuple(sorted([r["home"], r["away"]])), axis=1)
# residual from BOTH teams' perspective, venue-adjusted (resid_h already venue-aware via pred)
allm["resid_a"] = allm["away_goals"] - allm["pred_a"]

pooled = (allm.groupby("pair_key")
          .agg(n=("resid_h","size"),
               mean_resid=("resid_h","mean"),
               std_resid=("resid_h","std"))
          .reset_index())
pooled = pooled[pooled["n"] >= 10]
print("A — Pooled rivalries with >=10 meetings:", len(pooled))
print(f"    Std of pooled residuals: {pooled['mean_resid'].std():.3f}")
print(f"    Noise floor:             {allm['resid_h'].std()/np.sqrt(pooled['n'].mean()):.3f}")

# --- C: significance test per pairing (t-stat = mean / standard error) ---
pooled["se"] = pooled["std_resid"] / np.sqrt(pooled["n"])
pooled["t_stat"] = pooled["mean_resid"] / pooled["se"]
# How many pairings are "significant" (|t|>2) vs how many we'd expect by chance
n_sig = (pooled["t_stat"].abs() > 2).sum()
print(f"\nC — Pairings with |t| > 2: {n_sig} of {len(pooled)}")
print(f"    Expected by chance (~5%): {0.05*len(pooled):.1f}")

# --- D: does a pairing's residual in one half of history predict the other half? ---
allm_sorted = allm.sort_values("date_parsed")
half = allm_sorted["date_parsed"].quantile(0.5)
early = allm_sorted[allm_sorted["date_parsed"] <= half]
late  = allm_sorted[allm_sorted["date_parsed"] >  half]

e = early.groupby("pair_key")["resid_h"].mean().rename("early")
l = late.groupby("pair_key")["resid_h"].mean().rename("late")
persist = pd.concat([e, l], axis=1).dropna()
persist = persist[persist.index.map(lambda k: (early["pair_key"]==k).sum() >= 3 and (late["pair_key"]==k).sum() >= 3)]

corr = persist["early"].corr(persist["late"])
print(f"\nD — Does early-history pairing residual predict late-history?")
print(f"    Correlation: {corr:.3f}  (n={len(persist)} pairings)")
print(f"    ~0 = noise (no persistent effect) | strongly positive = real effect")

A — Pooled rivalries with >=10 meetings: 175
    Std of pooled residuals: 0.297
    Noise floor:             0.310

C — Pairings with |t| > 2: 10 of 175
    Expected by chance (~5%): 8.8

D — Does early-history pairing residual predict late-history?
    Correlation: -0.081  (n=148 pairings)
    ~0 = noise (no persistent effect) | strongly positive = real effect


In [24]:
# === Fixture-pairing effects within the RECENT era only (last 3 seasons) ===
recent_seasons = ["2022-23", "2023-24", "2024-25"]
rec = allm[allm["season"].isin(recent_seasons)].copy()

print("Recent-era matches:", len(rec))

# Pooled unordered rivalries, recent era only
rec["pair_key"] = rec.apply(lambda r: tuple(sorted([r["home"], r["away"]])), axis=1)
pr = (rec.groupby("pair_key")
      .agg(n=("resid_h","size"), mean_resid=("resid_h","mean"), std_resid=("resid_h","std"))
      .reset_index())
pr = pr[pr["n"] >= 4]   # >=4 meetings in 3 seasons = a regular fixture (both stayed up)

print("Rivalries with >=4 recent meetings:", len(pr))
print(f"\nSpread of pairing residuals: {pr['mean_resid'].std():.3f}")
print(f"Noise floor:                 {rec['resid_h'].std()/np.sqrt(pr['n'].mean()):.3f}")

# Significance count
pr["t"] = pr["mean_resid"] / (pr["std_resid"] / np.sqrt(pr["n"]))
print(f"\nPairings with |t|>2: {(pr['t'].abs()>2).sum()} of {len(pr)}  (chance ~{0.05*len(pr):.1f})")

print("\nStrongest recent pairing residuals:")
print(pr.reindex(pr["mean_resid"].abs().sort_values(ascending=False).index)
        .head(8)[["pair_key","n","mean_resid"]].to_string(index=False))

Recent-era matches: 1140
Rivalries with >=4 recent meetings: 171

Spread of pairing residuals: 0.567
Noise floor:                 0.540

Pairings with |t|>2: 27 of 171  (chance ~8.6)

Strongest recent pairing residuals:
                     pair_key  n  mean_resid
     (Aston Villa, Newcastle)  6    1.786113
(Aston Villa, Crystal Palace)  6    1.514721
       (Liverpool, Tottenham)  6    1.459403
       (Newcastle, Tottenham)  6    1.360044
        (Brighton, Leicester)  4    1.240209
       (Leicester, Tottenham)  4    1.225244
          (Brighton, Chelsea)  6    1.203910
     (Liverpool, Southampton)  4    1.202122
